In [2]:
import requests
from bs4 import BeautifulSoup
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
import numpy as np

In [3]:
url = "https://www.gutenberg.org/cache/epub/84/pg84-images.html"

response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

text = soup.get_text()

text = text.lower()
text = text.replace("\n", " ")

words = text.split()

print("Total words:", len(words))

Total words: 78110


In [4]:
vocab = Counter(words)
word2idx = {word:i for i,(word,_) in enumerate(vocab.items())}
idx2word = {i:word for word,i in word2idx.items()}

tokens = [word2idx[w] for w in words]

vocab_size = len(word2idx)
print("Vocabulary size:", vocab_size)

Vocabulary size: 11677


In [5]:
window_size = 100
seq_len = window_size - 1

data = []
targets = []

for i in range(len(tokens) - window_size):

    seq = tokens[i:i+seq_len]
    target = tokens[i+seq_len]

    data.append(seq)
    targets.append(target)

data = torch.tensor(data)
targets = torch.tensor(targets)

print(data.shape)
print(targets.shape)

torch.Size([78010, 99])
torch.Size([78010])


In [6]:
class TextLSTM(nn.Module):

    def __init__(self, vocab_size, embed_dim=50, hidden_dim=128):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):

        x = self.embedding(x)
        output, _ = self.lstm(x)

        last_output = output[:, -1, :]

        out = self.fc(last_output)

        return out

In [7]:
model = TextLSTM(vocab_size)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [8]:
epochs = 5
batch_size = 64

for epoch in range(epochs):

    total_loss = 0

    for i in range(0, len(data), batch_size):

        x = data[i:i+batch_size]
        y = targets[i:i+batch_size]

        optimizer.zero_grad()

        outputs = model(x)

        loss = criterion(outputs, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print("Epoch:", epoch+1, "Loss:", total_loss)

Epoch: 1 Loss: 8552.918228626251
Epoch: 2 Loss: 7480.397133350372
Epoch: 3 Loss: 6913.100268363953
Epoch: 4 Loss: 6447.774819850922
Epoch: 5 Loss: 6037.7454080581665


In [9]:
def generate_text(seed_text, num_words=100):

    model.eval()

    seed_tokens = [word2idx[w] for w in seed_text.lower().split() if w in word2idx]

    for _ in range(num_words):

        input_seq = torch.tensor(seed_tokens[-seq_len:]).unsqueeze(0)

        with torch.no_grad():
            output = model(input_seq)

        next_token = torch.argmax(output).item()

        seed_tokens.append(next_token)

    generated_words = [idx2word[i] for i in seed_tokens]

    return " ".join(generated_words)

In [10]:
seed = "the monster was"

generated = generate_text(seed, 100)

print(generated)

the monster was incomplete. with the project gutenberg™ gutenberg™ electronic work you may be located to provide the project gutenberg literary archive foundation in the project gutenberg literary archive foundation to the project gutenberg literary archive foundation in the united states, of the project gutenberg literary archive foundation in the project gutenberg literary archive foundation in the united states. states of the project gutenberg literary archive foundation in the project gutenberg literary archive foundation in the united states. states of paragraph 1.f.3, of the project gutenberg literary archive foundation you may be located with the project gutenberg literary archive foundation in the united
